In [7]:
import numpy as np, torch

from src.utils import DoubleIntegrator2D, CustomLogger, solve_and_plot_ipopt
from src.sindy import CompiledFunctionLibrary, SINDyVectorized
from src.trainer import SparseTrainer, SparsePolicyBuilder2D
from src.data import get_policy_data
from src.plot import *
# Set random seed for reproducibility
torch.manual_seed(0)

# Use GPU if available, otherwise fallback to CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


NameError: name 'SINDyVectorized' is not defined

In [18]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import os, glob, torch

run_id = 1
models_dir = f"results/tests/dynamics/run_{run_id}/saved_models"   # change root here
ckpt = torch.load(str(models_dir) + "/sindy.pt", map_location=device)

# Rebuild library and model skeleton
lib = CompiledFunctionLibrary(**ckpt["lib_cfg"])
sindy = SINDyVectorized(
    library=lib,
    n_out=ckpt["model_cfg"]["n_out"],
    device=device,
)

# Install pruning structure BEFORE loading weights (so shapes match)
sindy.active_idx = [list(map(int, s)) for s in ckpt["active_idx"]]

# Resize Xi ParameterList to match saved shapes
for i, idxs in enumerate(sindy.active_idx):
    sindy.Xi[i] = nn.Parameter(torch.zeros(len(idxs), 1, device=device), requires_grad=True)

# Recompile the fast plan for the current union of terms
sindy._plan = sindy._recompile_fast_path(device=device)

# Load weights
sindy.load_state_dict(ckpt["state_dict"], strict=True)


/var/folders/7d/4w2xzdbj279fj424x2h2w2xh0000gn/T/ipykernel_5082/3737867106.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(models_dir) + "/sindy.p

<All keys matched successfully>

In [20]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import os, glob, torch

run_id = 2
models_dir = f"results/tests/sparse/run_{run_id}/SINDy/saved_models"   # change root here
ckpt = torch.load(str(models_dir) + "/policy_sparse.pt", map_location=device)

# Rebuild library and model skeleton
policy_lib = CompiledFunctionLibrary(**ckpt["lib_cfg"])
policy_sparse = SINDyVectorized(
    library=policy_lib,
    n_out=ckpt["model_cfg"]["n_out"],
    policy_name=ckpt["model_cfg"]["policy_name"],
    device=device,
)

# Install pruning structure BEFORE loading weights (so shapes match)
policy_sparse.active_idx = [list(map(int, s)) for s in ckpt["active_idx"]]

# Resize Xi ParameterList to match saved shapes
for i, idxs in enumerate(policy_sparse.active_idx):
    policy_sparse.Xi[i] = nn.Parameter(torch.zeros(len(idxs), 1, device=device), requires_grad=True)

# Recompile the fast plan for the current union of terms
policy_sparse._plan = policy_sparse._recompile_fast_path(device=device)

# Load weights
policy_sparse.load_state_dict(ckpt["state_dict"], strict=True)


/var/folders/7d/4w2xzdbj279fj424x2h2w2xh0000gn/T/ipykernel_5082/854369725.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(models_dir) + "/policy_s

<All keys matched successfully>

In [21]:
policy_sparse.pretty_print()

u0 = -5.4323e-01·x0 + +1.8101e-01·r1 + +3.5956e-01·sqrt(x0) + -2.2505e-01·sqrt(x3) + -6.6320e-01·x0*r1 + -1.4094e+00·x1*r1 + +1.3210e+00·x2*r0 + +1.1133e+00·x3*r0
u1 = +1.5959e-01·x1 + -6.5742e-01·x2 + -1.0599e+00·x3 + +6.5525e-01·r1 + +1.7086e-01·sqrt(x0)
